In [103]:
# ! pip install torchao

In [104]:
# get timestamp today
import time
import datetime

today = datetime.date.today()
timestamp_today = time.mktime(today.timetuple())
print("Timestamp today:", timestamp_today)

# get all hourly timestamps for today and convert to torch tensor
import torch
import numpy as np

timestamps = np.arange(timestamp_today, timestamp_today + 86400, 3600)
t = torch.tensor(timestamps)
print("Timestamps for today:", t)

Timestamp today: 1730908800.0
Timestamps for today: tensor([1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09,
        1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09, 1.7309e+09,
        1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09,
        1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09, 1.7310e+09],
       dtype=torch.float64)


In [105]:
# convert t to positional encodings without any libraries
def get_positional_encodings(t: torch.Tensor, d_model: int) -> torch.Tensor:
    # calculate the positional encodings
    pe = torch.zeros(t.size(0), d_model)
    for pos in range(d_model):
        for i in range(t.size(0)):
            pe[i, pos] = t[i] / 10000**(2*pos/d_model)
    return pe

d_model = 105
pe = get_positional_encodings(t, d_model)

# normalize pe from -1 to 1
pe = 2 * (pe - pe.min()) / (pe.max() - pe.min()) - 1
pe

tensor([[ 0.9999,  0.6781,  0.4081,  ..., -1.0000, -1.0000, -1.0000],
        [ 0.9999,  0.6781,  0.4081,  ..., -1.0000, -1.0000, -1.0000],
        [ 0.9999,  0.6781,  0.4081,  ..., -1.0000, -1.0000, -1.0000],
        ...,
        [ 1.0000,  0.6782,  0.4081,  ..., -1.0000, -1.0000, -1.0000],
        [ 1.0000,  0.6782,  0.4081,  ..., -1.0000, -1.0000, -1.0000],
        [ 1.0000,  0.6782,  0.4082,  ..., -1.0000, -1.0000, -1.0000]])

In [106]:
# coutn the number of parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

tensor([[-4.0968e-01,  1.2520e+00, -4.5313e-01,  ...,  6.2545e-01,
         -2.1454e-01,  8.1117e-01],
        [ 2.5495e-01, -5.1476e-01,  2.7204e-01,  ..., -8.3973e-01,
         -6.3196e-01,  2.6793e-01],
        [-8.4198e-04, -3.5383e-02, -1.4672e+00,  ...,  1.8257e+00,
          2.1076e+00,  8.8008e-01],
        ...,
        [ 8.9428e-02,  3.4731e-03, -3.0112e-01,  ..., -5.5458e-01,
         -2.5902e-01,  4.7853e-01],
        [ 1.0775e+00,  2.3183e-01, -1.0524e+00,  ..., -1.7103e+00,
         -9.5364e-01, -1.1811e+00],
        [-6.1732e-01,  1.8050e+00,  3.5672e-01,  ..., -1.7490e-02,
          1.2799e-01, -4.5105e-01]])

In [137]:
# generate random tensor with gaussian distribution with size 24x105
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

x = torch.randn(24, d_model)
x_pe = x + pe
x_pe = x_pe.to(device)
x_pe = x_pe.unsqueeze(0)

import torch.nn as nn
from einops import rearrange

x_pe.shape


torch.Size([1, 24, 105])

In [128]:
from mamba_ssm import Mamba2

In [183]:
# Create a variational autoencoder

class S6_VAE(nn.Module):
    def __init__(self):
        super(S6_VAE, self).__init__()

        self.encoder = nn.ModuleList([
            nn.Conv1d(in_channels=105, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            Mamba2(d_model=64, headdim=64//8),
            nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            Mamba2(d_model=32, headdim=32//8),
            nn.Conv1d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            Mamba2(d_model=16, headdim=16//8),
        ])

        self.decoder = nn.ModuleList([
            nn.ConvTranspose1d(in_channels=8, out_channels=105, kernel_size=3, padding=1),
            nn.ReLU(),
            Mamba2(d_model=105, headdim=105, expand=8),
        ])

        self.fc1 = nn.Linear(16, 8)
        self.fc2 = nn.Linear(16, 8)

    def encode(self, x):
        for i in range(len(self.encoder)//3):
            x = rearrange(x,'b l c -> b c l')
            x = self.encoder[i*3+0](x) #conv
            # print('after conv', x.shape)
            x = self.encoder[i*3+1](x) #non-linearity
            x = rearrange(x, 'b c l -> b l c') 
            x = self.encoder[i*3+2](x) #state-space
            # print('after ssm', x.shape)

        mu, logvar = self.fc1(x), self.fc2(x)
        return mu, logvar


    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    
    def decode(self, z):
        z = rearrange(z,'b l c -> b c l')
        z = self.decoder[0](z) #conv
        z = self.decoder[1](z) #non-linearity
        z = rearrange(z, 'b c l -> b l c')
        z = self.decoder[2](z)
        return z
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [191]:
tsg = S6_VAE().to(device)
x_hat, mu,logvar = tsg(x_pe)
x_hat - x_pe


after conv torch.Size([1, 105, 24])
after rearrange torch.Size([1, 24, 105])


tensor([[[-2.3775, -0.9226,  0.2131,  ...,  1.7955,  2.5650,  1.5491],
         [-0.6891, -0.3272, -0.7091,  ...,  1.5528,  3.7560,  2.1781],
         [-1.1886, -1.0825, -1.0724,  ...,  1.2121,  0.4276,  2.6640],
         ...,
         [-1.5219, -1.0677,  0.4126,  ...,  2.4283,  1.4515,  0.5314],
         [ 0.9935, -1.0898, -0.0218,  ...,  0.2894,  2.3323,  0.7651],
         [-0.9233, -0.0927,  0.8209,  ...,  2.5638,  1.5701, -0.3390]]],
       device='cuda:0', grad_fn=<SubBackward0>)

AssertionError: 

In [145]:
9//3

3

300392

tensor([[1.7309e+09, 1.4524e+09, 1.2187e+09,  ..., 2.9299e+01, 2.4584e+01,
         2.0628e+01],
        [1.7309e+09, 1.4524e+09, 1.2187e+09,  ..., 2.9299e+01, 2.4584e+01,
         2.0628e+01],
        [1.7309e+09, 1.4524e+09, 1.2187e+09,  ..., 2.9299e+01, 2.4584e+01,
         2.0628e+01],
        ...,
        [1.7310e+09, 1.4525e+09, 1.2187e+09,  ..., 2.9300e+01, 2.4585e+01,
         2.0629e+01],
        [1.7310e+09, 1.4525e+09, 1.2187e+09,  ..., 2.9300e+01, 2.4585e+01,
         2.0629e+01],
        [1.7310e+09, 1.4525e+09, 1.2187e+09,  ..., 2.9300e+01, 2.4585e+01,
         2.0629e+01]])